In [ ]:
#  MÉMO DE DÉMARRAGE :

#  1. Se connecter à la Raspberry Pi (ssh jubilee@10.0.9.55) mdp : projet_indus

#  2. Lancer le serveur avec la commande : python3 serveur_pi.py

#  3. Vérifier que l'IP est bien 10.0.9.55 et le port 5001

In [5]:
from science_jubilee.JubileeManager import JubileeManager
from science_jubilee.decks.LoadAll import load_all
from science_jubilee.tools.HTTPSyringe import *
from science_jubilee.labware.Labware import *
from science_jubilee.decks.Deck import *

# Connexion à la Jubilee
jm = JubileeManager(address="10.0.9.6", simulated=False)
ser = HTTPSyringe

# Homing
#jm.controller.home_all()

# Descendre le plateau pour positionner le deck
#jm.controller.move_to(z=300)
#jm.controller.move_to(x=150, y=150)

# Enregistrer le deck
deck = jm.load_deck("test1")
load_all(jm,"test1")

#affiche les slots
jm.deck.list_slots()
jm.deck.get_all_slot_machine_coordinates()

jm.status()

#jm.controller.move_to(x=150, y=150)
#jm.controller.move_to(z=50)

# 1. On récupère les infos de ton outil par son nom
dict_outil = jm.get_tool_by_name("Pipette")
idx = dict_outil['index']

# 2. On prépare l'outil physiquement
#jm.controller.pickup_tool_sequence(idx)
jm.set_active_tool(idx)
jm.set_tool_offset(idx, (0, -31.5, 35))

tool1 = jm.tools_list[idx]
tool1.__class__ = ser

tool1.url_materiel = "http://10.0.9.55:5001" 
tool1._init_gpio()

tool1._machine = jm.controller  # On le lie au contrôleur pour les mouvements
tool1.is_active_tool = True     # On valide le décorateur de sécurité

# 4. CORRECTIF : On crée la méthode safe_z_movement si elle manque
if not hasattr(jm.controller, 'safe_z_movement'):
    # On remonte à 80mm de hauteur pour ne rien toucher
    tool1.safe_z_movement = lambda: jm.controller.move_to(z=220)
    print("✅ Correctif safe_z_movement appliqué.")

print(f"✅ Outil '{tool1.name}' prêt (Classe: {type(tool1).__name__})")

slot_1 = jm.deck.get_slot('1')
print(slot_1)
well_source = slot_1.labware.wells['A1']
well_source.y = well_source.y + slot_1.coordinates[1] - 50 # ac offset de x
well_source.x = well_source.x + slot_1.coordinates[0] - 5.5 #ac offset de y
print(well_source.x)
print(well_source.y)

2026-04-07 17:24:44 - [JubileeController]  - INFO - Initializing JubileeController (simulated=False, address=10.0.9.6)
2026-04-07 17:24:44 - [JubileeController]  - WARNING - Disconnecting this application from the network will halt connection to Jubilee.
2026-04-07 17:24:44 - [JubileeController]  - INFO - Connecting to Jubilee machine...
2026-04-07 17:24:44 - [JubileeController]  - INFO - Successfully connected and initialized Jubilee machine.
2026-04-07 17:24:44 - [JubileeManager]     - INFO - JubileeManager initialized (simulated=False, max_tools=4).
2026-04-07 17:24:44 - [Deck]               - INFO - Loading deck configuration from: /home/nicolas/Documents/ROB-4/Projet_indu/science-jubilee/src/science_jubilee/decks/deck_definition/test1.json
2026-04-07 17:24:44 - [Deck]               - INFO - Deck 'Experience1' loaded with 2 slots.
2026-04-07 17:24:44 - [JubileeManager]     - INFO - Deck 'Experience1' loaded from '/home/nicolas/Documents/ROB-4/Projet_indu/science-jubilee/src/science

[Pipette] ✅ Connecté au serveur matériel du Raspberry Pi.
✅ Correctif safe_z_movement appliqué.
✅ Outil 'Pipette' prêt (Classe: HTTPSyringe)
Slot(slot_index='1', coordinates=(110.05, 25.12), shape='rectangle', width=127.76, length=85.57, diameter=None, has_labware=True, labware=reservoir: agilent_1_reservoir_290ml)
168.43
17.905


In [8]:
# offset pipette
# x = -5.5
# y = -50
# z = 121 pour vider

cpt = 0 
# --- CONFIGURATION DES PUITS ---
rows = ['A', 'B', 'C', 'D']
cols = range(1, 7)

# Si vous voulez juste tester sur 2 puits, décommentez les lignes ci-dessous :
rows = ['A']
cols = range(1, 3)

# On définit la SOURCE (ici slot A1 du slot 0)
slot_0 = jm.deck.get_slot('0')

jm.controller.move_to(x=150, y=150)
jm.controller.move_to(z=220)

print(f"🚀 Début de la distribution depuis {well_source.name}...")

# --- LA BOUCLE DE DISTRIBUTION ---
for r in rows:
    for c in cols:
        if ( cpt == 0 ):
            jm.controller.move_to(z=220)
            jm.controller.move_to(x=well_source.x,y=well_source.y)
            jm.controller.move_to(z=110)
            try:
                jm.controller.dwell(3000)
                tool1.remplir_seringue(temps_secondes=10.0)
            except Exception as e:
                print(f"❌ Erreur sur le puits {well_dest_name} : {e}")
                break
            jm.controller.move_to(z=220)
            cpt = 3
            
        well_dest_name = f"{r}{c}" # Génère A1, A2... D6
        
        # On récupère l'objet Well de destination
        well_dest = slot_0.labware.wells[well_dest_name]
        well_dest.y = (well_dest.y + slot_0.coordinates[1]) - 50
        well_dest.x = (well_dest.x + slot_0.coordinates[0]) - 5.5

        print(f"\n---> Transfert de {well_source.name} vers {well_dest_name}...")

        jm.controller.move_to(z=220)
        jm.controller.move_to(x=well_dest.x,y=well_dest.y)
        jm.controller.move_to(z=121)
        try:
            jm.controller.dwell(3000)
            tool1.avancer_jusqu_au_seuil(seuil=1, timeout_sec=5)
        except Exception as e:
            print(f"❌ Erreur sur le puits {well_dest_name} : {e}")
            break
        cpt = cpt - 1
        jm.controller.move_to(z=220)

print("\n✅ Distribution terminée sur toute la plaque !")

🚀 Début de la distribution depuis A1...
[Pipette] Vidage en cours pour 4.0 sec...
[Pipette] Remplissage en cours pour 10.0 sec...
[Pipette] Remplissage terminé.

---> Transfert de A1 vers A1...
[Pipette] Ordre au Pi : Moteur forward (Attente seuil >= 1V)
[Pipette] Timeout atteint (5s).
[Pipette] Moteur arrêté.

---> Transfert de A1 vers A2...
[Pipette] Ordre au Pi : Moteur forward (Attente seuil >= 1V)
[Pipette] Seuil atteint (1.00V).
[Pipette] Moteur arrêté.

✅ Distribution terminée sur toute la plaque !


In [ ]:
print(well_source)
print(slot_0)
print(slot_0.labware.wells['A1'])

In [ ]:
print(cols)